# 学習した分類器を保存・読み込みする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_classification/save_sklearn_model.ipynb)

NLP クックブック 11-11 ／ テキスト分類

学習済みの scikit-learn の分類器 (前処理込み) をファイルに保存して、別のノートブックやアプリで使いたい。

In [1]:
import joblib
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

cats = ["rec.sport.baseball", "sci.space", "comp.graphics"]
train = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))
pipe = make_pipeline(TfidfVectorizer(stop_words="english", sublinear_tf=True), LogisticRegression(max_iter=1000))
pipe.fit(train.data, train.target)

joblib.dump({"model": pipe, "labels": train.target_names}, "classifier.joblib", compress=3)

['classifier.joblib']

In [2]:
import os
import sklearn

bundle = joblib.load("classifier.joblib")
model, labels = bundle["model"], bundle["labels"]
for text in ["The rocket reached orbit after launch.", "He hit a home run in the ninth inning."]:
    print(labels[model.predict([text])[0]], "|", text)
print(f"{os.path.getsize('classifier.joblib') / 1e6:.1f} MB, scikit-learn {sklearn.__version__}")

sci.space | The rocket reached orbit after launch.
rec.sport.baseball | He hit a home run in the ninth inning.
0.6 MB, scikit-learn 1.9.1


- ベクトライザと分類器を `Pipeline` にまとめて保存するのが鉄則。別々に保存すると、語彙と重みの対応がずれる事故が起きる。
- pickle 形式なので、読み込む側の scikit-learn のバージョンを揃える。また、信頼できないファイルを `load` してはいけない (任意コードが実行されうる)。
- Colab ではランタイムが切れるとファイルが消える。Google Drive に保存するか、ダウンロードしておく。